# Chapter 11. Deep Learning in Chemistry
## 11.1. Perceptrons, linear units, and gradients

A weighted sum is a useful building block, but the output rule determines what the model does. We will separate **hard-threshold classification**, **linear regression**, and **logistic classification**, then check a gradient and train a small PyTorch model.

### Learning objectives

- Distinguish a classical perceptron from an affine linear unit and a logistic unit.
- Calculate a squared-error gradient and verify it with automatic differentiation.
- Explain why one data point cannot identify several independent parameters.
- Use correct batch/feature/target shapes, training-only preprocessing, and held-out evaluation.
- Compare gradient-based fitting with least squares and save enough information for repeatable inference.

**Run independently:** use the course environment in the [README](Readme.md). These are explicitly **synthetic, dimensionless teaching data**, not measured chemical properties. Everything runs on one CPU thread; no GPU, installation cell, downloaded image, or interactive plotting backend is needed.

### A gentle entry point

A model takes an ordered list of **features** (input numbers) and predicts a **target** (the quantity we want). A **weight** says how strongly an input contributes; a **bias** supplies an offset. A **loss** measures disagreement with known targets. Training changes weights to reduce that disagreement on training examples.

| Symbol | Meaning in this lesson |
| --- | --- |
| $x_j$ | Feature $j$ for one example |
| $w_j$ | Learned coefficient for feature $j$ |
| $b$ | Learned offset |
| $\hat y$ and $y$ | Prediction and reference target |
| $L$ | Loss: a number we try to reduce |
| $\eta$ | Learning rate: size of a parameter update |

**Core route:** weighted sums and output rules → one gradient and its picture → a fitted linear model and baselines → the calculated chemistry check. The matrix derivatives, rank test, and checkpoint details are **deeper reading** after the input–prediction–error loop is clear. Ordinary arithmetic and the array-shape ideas in Chapter 1 are sufficient to start.

**Predict:** if a positive feature has an overly large positive weight, which way should that weight move to reduce an overprediction? The gradient example makes this direction explicit.

In [ ]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn

OUTPUT_DIR = Path("outputs/chapter11_part1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20261101
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cpu")
print(f"PyTorch {torch.__version__}; device={DEVICE}; CPU threads={torch.get_num_threads()}")

### 11.1.1. One weighted sum, three different models

First compute $z=\mathbf w^\mathsf T\mathbf x+b$. The bias makes this an **affine** map (commonly called a linear layer).

| Model | Output | Typical learning rule or loss |
|---|---|---|
| Classical binary perceptron | $\hat y=\mathbf 1[z\ge0]$ | Mistake-driven perceptron updates |
| Linear regression unit | $\hat y=z$ | Squared error / least squares |
| Logistic regression unit | $p=\sigma(z)=1/(1+e^{-z})$ | Binary cross-entropy |

The hard threshold is constant almost everywhere and discontinuous at zero. Ordinary backpropagation through that threshold does not give the classical perceptron learning rule. For binary logistic training, PyTorch's `BCEWithLogitsLoss` takes the **raw logits** $z$ and combines sigmoid with the loss stably; do not apply sigmoid a second time before it. A sigmoid score has a probabilistic interpretation under the model, but reliable calibration still needs assessment.

`nn.Linear(d, q)` itself applies only an affine transformation. It does not include thresholding, sigmoid, or any other activation. See the [Linear API](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) and [BCEWithLogitsLoss API](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html).

In [ ]:
z = np.linspace(-3, 3, 301)
outputs = [z, 1/(1+np.exp(-z)), (z >= 0).astype(float)]
titles = ["Linear output", "Logistic output", "Hard-threshold output"]
fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
for ax, values, title in zip(axes, outputs, titles):
    ax.plot(z, values, color="#287fa3")
    ax.set(xlabel="Weighted sum z", ylabel="Output", title=title)
    ax.grid(alpha=0.2)
plt.show()

### 11.1.2. A real hard-threshold perceptron

For labels in $\{0,1\}$, a mistake-driven update is

$$
\mathbf w\leftarrow\mathbf w+\eta(y-\hat y)\mathbf x,\qquad
b\leftarrow b+\eta(y-\hat y).
$$

The four-row AND truth table is linearly separable. We learn it with a small, explicitly bounded loop. This is a demonstration of a logical rule on its complete truth table, **not** a held-out generalization experiment. The perceptron convergence result requires linearly separable data; a single affine boundary cannot represent XOR. The classic perceptron was introduced by [Rosenblatt (1958)](https://doi.org/10.1037/h0042519).

In [ ]:
logic_x = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
and_y = np.array([0, 0, 0, 1])
perceptron_w = np.zeros(2)
perceptron_b = 0.0
mistakes_per_pass = []
for pass_index in range(20):
    mistakes = 0
    for row, target in zip(logic_x, and_y):
        predicted = int(row @ perceptron_w + perceptron_b >= 0)
        error = int(target) - predicted
        perceptron_w += error * row  # learning rate = 1
        perceptron_b += error
        mistakes += int(error != 0)
    mistakes_per_pass.append(mistakes)
    if mistakes == 0:
        break
and_prediction = (logic_x @ perceptron_w + perceptron_b >= 0).astype(int)
np.testing.assert_array_equal(and_prediction, and_y)
print(f"AND learned in {len(mistakes_per_pass)} passes; w={perceptron_w}, b={perceptron_b}")
display(pd.DataFrame({"x1": logic_x[:, 0], "x2": logic_x[:, 1],
                      "AND target": and_y, "perceptron prediction": and_prediction}))

### 11.1.3. Check a linear unit's gradient

For $N$ rows and one target per row,

$$
\hat{\mathbf y}=X\mathbf w+b,\qquad
L=\frac1N\sum_{i=1}^N(\hat y_i-y_i)^2,
$$
$$
\nabla_{\mathbf w}L=\frac2N X^\mathsf T(\hat{\mathbf y}-\mathbf y),\qquad
\frac{\partial L}{\partial b}=\frac2N\sum_i(\hat y_i-y_i).
$$

Automatic differentiation applies the chain rule to the operations used in the forward calculation. Here the derivative is also simple enough to calculate directly. We compare three independent routes: these analytic expressions, central finite differences, and PyTorch autograd.

Use $\mathbf x=(3.4,2.4,-1.7)$, $y=-7.2$, initial $\mathbf w=(0.3,-0.5,0.2)$, and $b=0.9$. The prediction is **0.38**, the residual is **7.58**, and the exact squared loss is **57.4564**. Double precision makes this numerical gradient check easier to interpret. See [PyTorch autograd](https://docs.pytorch.org/docs/stable/notes/autograd.html).

In [ ]:
x_demo = torch.tensor([[3.4, 2.4, -1.7]], dtype=torch.float64)
y_demo = torch.tensor([[-7.2]], dtype=torch.float64)
theta = torch.tensor([0.3, -0.5, 0.2, 0.9], dtype=torch.float64, requires_grad=True)

def demo_loss(parameters):
    prediction = x_demo @ parameters[:3].reshape(3, 1) + parameters[3]
    assert prediction.shape == y_demo.shape == (1, 1)
    return (prediction-y_demo).square().mean()

loss = demo_loss(theta)
loss.backward()
residual = float((x_demo @ theta[:3].detach().reshape(3, 1) + theta[3].detach() - y_demo).item())
analytic_gradient = 2 * residual * np.r_[x_demo.numpy().ravel(), 1.0]
parameters_numpy = theta.detach().numpy().copy()
def numpy_loss(parameters):
    return float((x_demo.numpy().ravel() @ parameters[:3] + parameters[3] + 7.2)**2)
step = 1e-6
finite_difference = np.array([
    (numpy_loss(parameters_numpy + step*np.eye(4)[i]) -
     numpy_loss(parameters_numpy - step*np.eye(4)[i])) / (2*step)
    for i in range(4)
])
np.testing.assert_allclose(loss.item(), 57.4564, atol=1e-10)
np.testing.assert_allclose(theta.grad.numpy(), analytic_gradient, atol=1e-10)
np.testing.assert_allclose(finite_difference, analytic_gradient, rtol=1e-7, atol=1e-7)
assert torch.autograd.gradcheck(demo_loss, (theta,), eps=1e-6, atol=1e-5)
display(pd.DataFrame({"parameter": ["w1", "w2", "w3", "b"],
                      "analytic": analytic_gradient, "finite difference": finite_difference,
                      "autograd": theta.grad.numpy()}).round(6))

In [ ]:
learning_rate = 0.001
updated_theta = theta.detach() - learning_rate * theta.grad
updated_loss = demo_loss(updated_theta).item()
assert updated_loss < loss.item()
print("Updated [w1, w2, w3, b]:", updated_theta.numpy())
print(f"One gradient step: loss {loss.item():.6f} → {updated_loss:.6f}")

# Different parameters can give exactly the same prediction on this one row.
alternative_theta = theta.detach() + torch.tensor([1., 0., 0., -3.4], dtype=torch.float64)
torch.testing.assert_close(demo_loss(alternative_theta), loss.detach())
single_row_design = np.column_stack([x_demo.numpy(), np.ones(1)])
print(f"One-row design rank: {np.linalg.matrix_rank(single_row_design)} for 4 parameters")

### See why one example leaves a whole valley of solutions

The previous calculation has four parameters. To draw a surface, **hold $w_2=-0.5$ and $w_3=0.2$ fixed** and vary only $w_1$ and $b$. The one-row residual is then $3.4w_1+b+5.66$. Every point on the line $b=-3.4w_1-5.66$ has zero loss for this row.

The arrow below is a small downhill step using **only those two coordinates**, so it is not the four-parameter update above. The gradient is perpendicular to a contour of constant loss. A zero-loss valley is not evidence that all its parameter choices predict new inputs equally well.

In [ ]:
w1_grid = np.linspace(-2.4, 1.0, 180)
bias_grid = np.linspace(-3.0, 3.0, 180)
W1, BIAS = np.meshgrid(w1_grid, bias_grid)
loss_slice = (3.4*W1 + BIAS + 5.66)**2
start_2d = np.array([0.3, 0.9])
gradient_2d = 2*(3.4*start_2d[0]+start_2d[1]+5.66)*np.array([3.4, 1.0])
next_2d = start_2d - 0.01*gradient_2d
assert (3.4*next_2d[0]+next_2d[1]+5.66)**2 < numpy_loss(parameters_numpy)
fig, ax = plt.subplots(figsize=(7, 5.5), layout="constrained")
contours = ax.contour(W1, BIAS, loss_slice, levels=[1, 5, 15, 40, 80, 120], colors="#64748b")
ax.clabel(contours, inline=True, fontsize=8)
ax.plot(w1_grid, -3.4*w1_grid-5.66, color="#b45309", label="zero-loss parameter choices")
ax.scatter(*start_2d, color="#b91c1c", zorder=3)
ax.annotate("", xy=next_2d, xytext=start_2d, arrowprops={"arrowstyle": "->", "lw": 2, "color": "#b91c1c"})
ax.text(start_2d[0]+0.06, start_2d[1]+0.18, "start", color="#b91c1c")
ax.set(xlabel="Weight w1 (w2 and w3 fixed)", ylabel="Bias b",
       xlim=(w1_grid[0], w1_grid[-1]), ylim=(bias_grid[0], bias_grid[-1]),
       title="One-row loss: many parameter choices fit equally well", aspect="equal")
ax.legend(loc="lower left", fontsize=9)
plt.show()

**Explain:** moving along the orange line changes the parameters while preserving the one observed prediction. More copies of that identical input do not remove the ambiguity; additional informative inputs are needed. In chemical data, highly correlated descriptors can similarly make fitted coefficients hard to interpret, even when predictions on familiar rows look acceptable.

**A low loss on one row does not identify the generating weights.** One scalar equation cannot uniquely determine four independent parameters. Even if repeated training makes this one residual zero, infinitely many parameter vectors fit it. Identifying a linear relationship requires sufficiently varied inputs and a full-rank design; strong feature collinearity can make coefficients unstable even with many rows.

### 11.1.4. Shapes and the training loop

For a batch of $N$ samples with $d$ input features, use `X.shape == (N, d)`. For one continuous target, use both predictions and targets with shape `(N, 1)`. `nn.Linear(d, 1)` stores its weight as `(1, d)` and computes `X @ weight.T + bias`.

Do not compare `(N, 1)` predictions with `(N,)` targets: broadcasting can create an unintended `(N, N)` residual array. `[x1, x2]` represents one two-feature sample; `[[x1, x2]]` retains the batch dimension.

A differentiable training step is: clear accumulated gradients, predict, compute loss, call `backward()`, then update parameters. An **epoch** is one pass through the training set; in this small full-batch example, each epoch contains one update. We do not claim that every update must reduce a noisy or nonconvex objective.

### 11.1.5. Fit a full-rank synthetic linear problem

Generate $y=2x_1-3x_2+0.5+\epsilon$, with Gaussian noise of standard deviation 0.05. Split 144 rows into **96 training**, **24 validation**, and **24 test** rows before preprocessing. Training data fit parameters and scaling. Validation data are available for model choices; test data are reserved for the final fixed-model comparison.

This example fixes its learning rate and 120 epochs in advance. Part 11.2 demonstrates validation-based early stopping. The split is an ordinary random synthetic-data split; it does not model generalization to new chemical scaffolds, laboratories, or assay conditions.

We standardize each input using its **training mean and standard deviation**, then reuse those values for every other row. Scaling often improves optimization conditioning; it is not a universal mathematical requirement that all inputs or outputs lie between 0 and 1. Targets here remain in their original dimensionless units.

In [ ]:
rng = np.random.default_rng(SEED)
X_raw = rng.uniform(-2, 2, size=(144, 2))
y_raw = 2*X_raw[:, [0]] - 3*X_raw[:, [1]] + 0.5 + rng.normal(0, 0.05, size=(144, 1))
train_index, validation_index, test_index = np.split(rng.permutation(len(X_raw)), [96, 120])
assert len(set(train_index) & set(test_index)) == 0
assert len(set(train_index) | set(validation_index) | set(test_index)) == len(X_raw)
x_mean = X_raw[train_index].mean(axis=0, keepdims=True)
x_scale = X_raw[train_index].std(axis=0, keepdims=True)
assert np.all(x_scale > 0)
X = torch.tensor((X_raw-x_mean)/x_scale, dtype=torch.float32, device=DEVICE)
y = torch.tensor(y_raw, dtype=torch.float32, device=DEVICE)
assert X.shape == (144, 2) and y.shape == (144, 1)
design_train = np.column_stack([np.ones(len(train_index)), X_raw[train_index]])
assert np.linalg.matrix_rank(design_train) == 3
print("Split sizes:", len(train_index), len(validation_index), len(test_index))
print("Training design rank: 3 for the bias and two feature weights")

In [ ]:
torch.manual_seed(SEED)
linear_model = nn.Linear(2, 1).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.08)
N_EPOCHS = 120
train_losses = []
start = perf_counter()
for epoch in range(N_EPOCHS):
    linear_model.train()
    optimizer.zero_grad(set_to_none=True)
    prediction = linear_model(X[train_index])
    assert prediction.shape == y[train_index].shape
    training_loss = criterion(prediction, y[train_index])
    training_loss.backward()
    optimizer.step()
    with torch.no_grad():
        train_losses.append(criterion(linear_model(X[train_index]), y[train_index]).item())
training_seconds = perf_counter()-start
assert np.isfinite(train_losses).all() and train_losses[-1] < train_losses[0]
print(f"{N_EPOCHS} full-batch updates: {training_seconds:.3f} s")

### 11.1.6. Compare with useful baselines

The **training mean** ignores all features. Ordinary least squares (OLS) fits the same affine model by solving a linear algebra problem, without a learning-rate loop. It is a valuable baseline and numerical check: a neural-network library should not receive credit for discovering a relationship that a simpler method already fits.

Both baselines use only training targets. We now evaluate the fixed models on validation and test rows, with MSE and MAE in original target units (MSE has squared units). This small single split estimates performance only for the synthetic sampling setup; it does not establish chemical predictive accuracy.

In [ ]:
ols_coefficients = np.linalg.lstsq(design_train, y_raw[train_index], rcond=None)[0]
mean_target = float(y_raw[train_index].mean())
linear_model.eval()
with torch.inference_mode():
    neural_predictions = linear_model(X).cpu().numpy()
ols_predictions = np.column_stack([np.ones(len(X_raw)), X_raw]) @ ols_coefficients
np.testing.assert_allclose(neural_predictions, ols_predictions, atol=2e-4, rtol=1e-4)
rows = []
for split_name, indices in [("validation", validation_index), ("test", test_index)]:
    for model_name, predictions in [("training mean", np.full_like(y_raw, mean_target)),
                                     ("OLS", ols_predictions), ("PyTorch linear", neural_predictions)]:
        residual = predictions[indices]-y_raw[indices]
        rows.append({"split": split_name, "model": model_name,
                     "MSE": float(np.mean(residual**2)), "MAE": float(np.mean(np.abs(residual)))})
metrics = pd.DataFrame(rows)
display(metrics.round(6))
raw_weights = linear_model.weight.detach().cpu().numpy().ravel()/x_scale.ravel()
raw_bias = float(linear_model.bias.item()-raw_weights @ x_mean.ravel())
display(pd.DataFrame({"parameter": ["bias", "x1 weight", "x2 weight"],
                      "generating value": [0.5, 2.0, -3.0],
                      "fitted PyTorch value": np.r_[raw_bias, raw_weights]}).round(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.7), layout="constrained")
axes[0].semilogy(np.arange(1, N_EPOCHS+1), train_losses, color="#287fa3")
axes[0].set(xlabel="Epoch", ylabel="Post-update training MSE", title="Bounded linear-model training")
axes[1].scatter(y_raw[test_index], neural_predictions[test_index], color="#287fa3", label="Test rows")
limits = [float(y_raw[test_index].min())-0.4, float(y_raw[test_index].max())+0.4]
axes[1].plot(limits, limits, "k--", label="Perfect prediction")
axes[1].set(xlabel="Observed synthetic y", ylabel="Predicted y", title="Held-out test predictions")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()

### 11.1.7. Save the model together with preprocessing

The learned weights act on **standardized features**, so weights alone do not define predictions on raw inputs. Save the architecture, feature order, preprocessing values, target convention, and parameter state together. This is an inference checkpoint; resuming an optimizer exactly would also require its state and relevant random-number state.

We load only the file just created locally, with `weights_only=True` and `map_location="cpu"`, reconstruct the architecture, and verify a round trip. Use files from trusted sources; the restricted loader is not a reason to accept arbitrary checkpoints. See the [official save/load guide](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).

In [ ]:
checkpoint_path = OUTPUT_DIR / "synthetic_linear.pt"
checkpoint = {"format_version": 1, "architecture": {"in_features": 2, "out_features": 1},
              "feature_names": ["synthetic_x1", "synthetic_x2"], "target": "dimensionless synthetic y; unscaled",
              "x_mean": torch.tensor(x_mean, dtype=torch.float32),
              "x_scale": torch.tensor(x_scale, dtype=torch.float32),
              "state_dict": linear_model.state_dict(), "seed": SEED,
              "torch_version": str(torch.__version__)}
torch.save(checkpoint, checkpoint_path)
loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert loaded["format_version"] == 1 and loaded["feature_names"] == ["synthetic_x1", "synthetic_x2"]
restored = nn.Linear(**loaded["architecture"])
restored.load_state_dict(loaded["state_dict"])
restored.eval()
raw_examples = torch.tensor([[0.0, 0.0], [1.0, -1.0]], dtype=torch.float32)
assert raw_examples.shape == (2, 2)
assert np.all(raw_examples.numpy() >= X_raw[train_index].min(axis=0))
assert np.all(raw_examples.numpy() <= X_raw[train_index].max(axis=0))
with torch.inference_mode():
    scaled_examples = (raw_examples-loaded["x_mean"])/loaded["x_scale"]
    restored_predictions = restored(scaled_examples)
    torch.testing.assert_close(restored_predictions, linear_model(scaled_examples))
display(pd.DataFrame({"x1": raw_examples[:, 0].numpy(), "x2": raw_examples[:, 1].numpy(),
                      "prediction": restored_predictions[:, 0].numpy()}))
metrics.to_csv(OUTPUT_DIR / "linear_metrics.csv", index=False)
print(f"Verified inference checkpoint: {checkpoint_path}")

### Research application: check a target that chemistry already determines

Before training a large model, check a simple **calculated** property whose answer follows from composition. Consider straight-chain saturated monohydric alcohols, from methanol through octan-1-ol. Their formula is $\mathrm C_n\mathrm H_{2n+2}\mathrm O$, so average molar mass is affine in carbon count $n$:

$$M(n)=nM_{\rm C}+(2n+2)M_{\rm H}+M_{\rm O}.$$

RDKit calculates masses from its atomic-weight convention. These eight values are **not measured masses or solubilities**. Fit ordinary least squares on $n=1,\ldots,5$ and check $n=6,7,8$, fixed in advance. The exact composition rule provides a stronger check than model accuracy alone. [RDKit descriptor documentation](https://www.rdkit.org/docs/source/rdkit.Chem.Descriptors.html).

**Predict:** what mass increment should adding one $\mathrm{CH_2}$ unit contribute? Why should the same one-feature model fail for a molecule outside this formula family?

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

alcohol_n = np.arange(1, 9)
alcohol_smiles = ["C"*int(count)+"O" for count in alcohol_n]
alcohol_molecules = [Chem.MolFromSmiles(text) for text in alcohol_smiles]
assert all(mol is not None for mol in alcohol_molecules)
calculated_mass = np.array([Descriptors.MolWt(mol) for mol in alcohol_molecules])
periodic_table = Chem.GetPeriodicTable()
carbon_mass, hydrogen_mass, oxygen_mass = [periodic_table.GetAtomicWeight(z) for z in (6, 1, 8)]
formula_mass = alcohol_n*carbon_mass + (2*alcohol_n+2)*hydrogen_mass + oxygen_mass
np.testing.assert_allclose(calculated_mass, formula_mass, atol=1e-10)
mass_design = np.column_stack([np.ones(len(alcohol_n)), alcohol_n])
mass_coefficients = np.linalg.lstsq(mass_design[:5], calculated_mass[:5], rcond=None)[0]
mass_predictions = mass_design @ mass_coefficients
np.testing.assert_allclose(mass_predictions[5:], calculated_mass[5:], atol=1e-10)
print(f"Fitted slope: {mass_coefficients[1]:.3f} g/mol per added carbon in this family")
print(f"Formula CH2 increment: {carbon_mass+2*hydrogen_mass:.3f} g/mol")
display(pd.DataFrame({"carbon_count": alcohol_n, "SMILES": alcohol_smiles,
    "calculated_molar_mass_g_mol": calculated_mass, "linear_prediction_g_mol": mass_predictions,
    "role": ["fit"]*5+["predeclared check"]*3}))
fig, ax = plt.subplots(figsize=(7, 3.8), layout="constrained")
ax.plot(alcohol_n, mass_predictions, color="0.5", label="Line fitted on n = 1 to 5")
ax.scatter(alcohol_n[:5], calculated_mass[:5], color="#2563eb", label="Calculated fitting examples")
ax.scatter(alcohol_n[5:], calculated_mass[5:], color="#b45309", marker="s", label="Calculated held-out examples")
ax.set(xlabel="Carbon count n in CnH(2n+2)O", ylabel="Average molar mass (g/mol)",
       title="A known chemical composition rule provides an exact baseline", xticks=alcohol_n)
ax.legend(fontsize=8)
plt.show()

**Research decision:** this target is already computable from a validated formula, so a learned mass predictor mainly checks the representation and workflow. Success outside the fitting carbon-count range follows here from a known composition law; it does not justify general extrapolation by a neural network.

**Guided exercise:** use acetone (`CC(=O)C`), which also has three carbons. **Selected answer:** it has two fewer hydrogens than propan-1-ol, so carbon count alone would overpredict its molar mass by about 2.016 g/mol under the same convention. A solubility predictor would require a different target, suitable measurements, and more informative inputs.

### Exercises

1. Is `nn.Linear(3, 1)` by itself a hard-threshold perceptron? What operation would turn its output into a class decision?
2. Explain why a single affine decision boundary can represent AND but cannot represent XOR.
3. Verify the four gradients for the single-row example by hand. Why is their sign based on prediction minus target?
4. Construct another weight/bias vector with the same prediction on the one-row example. Does fitting that row imply recovering $(1,-3,2,0)$?
5. For 32 samples and three features, state the shapes of inputs, a one-output weight matrix, predictions, and targets.
6. Why are the PyTorch linear model and OLS almost identical here? Would an extra linear layer without a nonlinear activation enlarge their function class?
7. Why must preprocessing be fitted only on training rows and included in an inference checkpoint? Does lying within each feature's observed range prove a chemically meaningful applicability domain?

<details><summary>Suggested answers</summary>

1. No; it returns an affine score. A threshold supplies a hard decision. Logistic probabilities instead use sigmoid, with an appropriate classification loss for training.
2. One line separates the AND-positive corner from the other three. XOR's two classes occupy opposite corners; their convex hulls intersect, preventing separation by one line.
3. The gradients are 51.544, 36.384, −25.772, and 15.16. Differentiating $(\hat y-y)^2$ with respect to $\hat y$ gives $2(\hat y-y)$.
4. Add any $a$ to $w_1$ and subtract $3.4a$ from the bias. The one prediction remains unchanged, so the generating parameters are not identifiable from that row.
5. `(32, 3)`, `(1, 3)`, `(32, 1)`, and `(32, 1)`.
6. They optimize the same unregularized affine squared-error problem; the short gradient loop has converged close to its least-squares solution. Composed affine layers remain affine.
7. Held-out rows must not affect learned transformations. Saved weights expect the original feature order and scaling. Marginal range checks do not account for feature combinations, chemical structure, or distribution shift.

</details>

### Connection to chemistry and references

For molecular work, numerical features might be descriptors or fingerprints, and the target might be a measured property with stated units and conditions. A model trained on noisy, correlated chemical observations needs more careful representations, group-aware splits, and error analysis than these synthetic examples. [Part 11.3](Chapter11_Part3.ipynb) makes that connection; [Part 11.2](Chapter11_Part2.ipynb) first introduces nonlinear networks and validation-based selection.

- [Rosenblatt's perceptron paper](https://doi.org/10.1037/h0042519).
- [PyTorch Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html), [MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html), and [autograd](https://docs.pytorch.org/docs/stable/notes/autograd.html).
- [PyTorch reproducibility guidance](https://docs.pytorch.org/docs/stable/notes/randomness.html): seeds and deterministic algorithms help within a fixed setup, but do not guarantee identical results across all releases and platforms.
- [PyTorch save/load guidance](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).